In [4]:
import sqlite3
import pandas as pd
from pathlib import Path

In [6]:
cwd = Path.cwd()
price_path = cwd / 'data' / 'Metro_median_sale_price_now_uc_sfrcondo_month.csv'
zhvi_path  = cwd / 'data' / 'Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month (2).csv'
redfin_path = cwd / 'data' / 'Redfin_Home_Price_Index.csv'

price_df = pd.read_csv(price_path)
zhvi_df  = pd.read_csv(zhvi_path)
redfin_df = pd.read_csv(redfin_path, encoding='utf-16', sep='\t')

In [8]:
conn = sqlite3.connect("housing_project.db")

merged.to_sql("merged", conn, if_exists="replace", index=False)
yoy_long.to_sql("yoy_long", conn, if_exists="replace", index=False)
comparison_yoy_df.to_sql("comparison_yoy", conn, if_exists="replace", index=False)
full.to_sql("full", conn, if_exists="replace", index=False)


NameError: name 'merged' is not defined

In [ ]:
# Median sale price and ZHVI change over time
query_avg_over_time = """
SELECT
    date,
    AVG(median_price) AS avg_median_price,
    AVG(zhvi)         AS avg_zhvi
FROM merged
GROUP BY date
ORDER BY date;
"""

avg_over_time = pd.read_sql_query(query_avg_over_time, conn)
print(avg_over_time.head())

In [ ]:
# Overall ZHVI bias
query_bias_overall = """
SELECT
    AVG( (zhvi - median_price) * 1.0 / median_price ) AS mean_rel_diff,
    COUNT(*) AS n_obs
FROM merged
WHERE median_price IS NOT NULL
  AND zhvi IS NOT NULL;
"""

bias_overall = pd.read_sql_query(query_bias_overall, conn)
print(bias_overall)

In [ ]:
# ZHVI Bias by State
query_bias_by_state = """
SELECT
    StateName AS State,
    AVG( (zhvi - median_price) * 1.0 / median_price ) AS mean_bias,
    COUNT(*) AS sample_size
FROM merged
WHERE median_price IS NOT NULL
  AND zhvi IS NOT NULL
  AND StateName IS NOT NULL
GROUP BY StateName
ORDER BY mean_bias DESC;
"""

state_bias_sql = pd.read_sql_query(query_bias_by_state, conn)
print(state_bias_sql.head())

In [ ]:
# Comparing Zillow YoY vs Redfin YoY
query_yoy_compare = """
SELECT
    date,
    AVG(median_price_yoy) AS median_yoy_mean,
    AVG(redfin_hpi_yoy)   AS redfin_yoy_mean,
    COUNT(*)              AS sample_count
FROM comparison_yoy
WHERE median_price_yoy IS NOT NULL
  AND redfin_hpi_yoy IS NOT NULL
GROUP BY date
ORDER BY date;
"""

yoy_compare_sql = pd.read_sql_query(query_yoy_compare, conn)
print(yoy_compare_sql.head())

In [ ]:
df = full.copy()
df = df[['zhvi', 'yoy_price_change', 'redfin_hpi_yoy', 'median_price']].dropna()

threshold = df['median_price'].median()
df['high_price'] = (df['median_price'] > threshold).astype(int)

X_class = df[['zhvi', 'yoy_price_change', 'redfin_hpi_yoy']]
y_class = df['high_price']
